# Backtesting basics: violations + Kupiec POF (LR test)

**Updated:** 2026-03-05  
**Goal:** build an end-to-end backtesting pipeline (rolling VaR → violations → Kupiec POF p-value) on sample PnL data.

## What we test
- **Violation** occurs when realized loss exceeds the rolling VaR threshold: 

  $$ I_t = \mathbf{1}_{\{L_t > \mathrm{VaR}_{\alpha,t}\}} $$

- **Kupiec POF** tests whether the observed violation rate matches the expected rate $1-\alpha$.

## 1) Load sample PnL and define loss
We use the project convention `loss = -pnl`.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df = df.set_index("date").sort_index()

pnl = df["pnl"]
loss = -pnl

df.head()


## 2) Rolling VaR, violations, and Kupiec POF test
Compute rolling historical VaR and define violations as `loss > VaR`. Then test whether the observed violation rate is consistent with the expected rate `1 - alpha`.

In [ ]:
from riskmetrics.var import rolling_historical_var
from riskmetrics.backtest import var_violations, kupiec_pof_test

alpha = 0.99
window = 3  # demo window; realistic is ~250 for daily trading

rvar = rolling_historical_var(pnl, window=window, alpha=alpha)
viol = var_violations(loss, rvar)
out = kupiec_pof_test(viol, alpha=alpha)

rvar.tail(8), viol.tail(8), out


## 3) Sanity check
Verify that the violation flags are correct by inspecting `loss` and `VaR`.

In [ ]:
aligned = pd.concat([loss.rename("loss"), rvar.rename("VaR")], axis=1).dropna()
aligned["violation"] = (aligned["loss"] > aligned["VaR"]).astype(int)
aligned

## Summary
- With `alpha=0.99` and `window=3`, we observe a high violation rate on this tiny sample (pipeline sanity check).
- Kupiec POF returns LR and a p-value for coverage testing.
- For meaningful backtesting, we need a longer time series (e.g., >= 250 observations) and realistic windows.

---

## Next
- Run the same pipeline on a longer PnL series.
- Add a compact backtest report (violation rate, Kupiec p-value) and a plot (loss vs rolling VaR with violation markers).